In [1]:
import os
os.chdir("../")
%pwd

'/mnt/e/Deep_Learning_Object_Detection/MLOPs/pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class OnnxModelConfig:
    onnx_dir:       Path
    onnx_int8_dir:  Path

@dataclass(frozen=True)
class TensorRTConfig:
    root_dir: Path
    out_dir:  Path
    image_size: int
    batch_size: int
    onnx: OnnxModelConfig

In [3]:
from pneumonia_segmentation.constants import *
from pneumonia_segmentation.utils.common import read_yaml, create_directories

class ConfigManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH, 
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
        
    def _model_slug(self) -> str:
        p = self.params.prepare_base_model_params
        return f"{p.model_name}_{p.encoder}"

    def get_tensorrt_config(self) -> TensorRTConfig:
        config = self.config.tensorRT_config
        params = self.params.tensorRT_params
        create_directories([config.root_dir])
        
        onnx_config = self.config.onnx_config
        root_dir = Path(config.root_dir)
        slug = self._model_slug()
        
        return TensorRTConfig(
            root_dir    = root_dir,
            out_dir     = root_dir / slug / "model.engine",
            image_size  = params.image_size,
            batch_size  = params.batch_size,
            onnx = OnnxModelConfig(
                onnx_dir      = Path(onnx_config.root_dir) / slug / "model.onnx",
                onnx_int8_dir = Path(onnx_config.root_dir) / slug / "model_int8.onnx"
            )
        )

In [6]:
import pycuda.driver as cuda
import pycuda.autoinit
import torch, sys, json, time, numpy as np
import tensorrt as trt, onnxruntime as ort
import segmentation_models_pytorch as smp

from pneumonia_segmentation import logging
from pneumonia_segmentation.exception import CustomException

class TensorRT:
    def __init__(self, config: TensorRTConfig):
        self.config    = config
        self.batch_size= self.config.batch_size
        self.img_size  = self.config.image_size
        self.onnx_path = self.config.onnx.onnx_dir
        self.engine_path = self.config.out_dir
        self.engine_path.parent.mkdir(parents=True, exist_ok=True)
        self.logger    = trt.Logger(trt.Logger.WARNING)

    def _measure(self, fn, n_runs: int = 50, warmup: int = 5, device = "cpu"):
        for _ in range(warmup):
            fn()
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_runs):
            fn()
        if device == 'cuda':
            torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n_runs * 1000  # ms
    
    def _trt_run(self, dummy_np):
        input_mem = cuda.mem_alloc(dummy_np.nbytes)
        output = np.empty((1, 1, self.img_size, self.img_size), dtype=np.float32)
        output_mem = cuda.mem_alloc(output.nbytes)
        cuda.memcpy_htod(input_mem, dummy_np)
        self.context.execute_v2([int(input_mem), int(output_mem)])
        cuda.memcpy_dtoh(output, output_mem)
    
    def quick_infer(self):
        runtime = trt.Runtime(self.logger)
        with open(self.engine_path, "rb") as f:
            engine = runtime.deserialize_cuda_engine(f.read())
        
        results  = {}
        dummy_np = np.random.randn(
            1, 3, self.img_size, self.img_size
        ).astype(np.float32)
        
        self.context = engine.create_execution_context()
        self.context.set_input_shape("input", dummy_np.shape)
        
        for _ in range(10): self._trt_run(dummy_np)
        
        results['tensorrt'] = self._measure(lambda: self._trt_run(dummy_np), device='cuda')
        print(f"TensorRT:     {results['tensorrt']:.2f} ms")
    
    def _create_parser(self) -> tuple:
        builder = trt.Builder(self.logger)
        network = builder.create_network(
            1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        )
        return builder, network, trt.OnnxParser(network, self.logger)
    
    def _create_config(self, builder, network) -> trt.IBuilderConfig:
        config  = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
        profile = builder.create_optimization_profile()
        profile.set_shape(
            network.get_input(0).name,
            min=(1, 3, self.img_size, self.img_size),
            opt=(1, 3, self.img_size, self.img_size),
            max=(self.batch_size, 3, self.img_size, self.img_size),
        )
        config.add_optimization_profile(profile)
        return config
    
    def _build_engine(self, onnx_path: str, engine_path: str):
        builder, network, parser = self._create_parser()
        
        with open(onnx_path, "rb") as f:
            if not parser.parse(f.read()):
                for error in range(parser.num_errors):
                    self.logger.info(parser.get_error(error))
                return None
        
        try:
            serialized_engine = builder.build_serialized_network(
                network, self._create_config(builder, network)
            )
        except Exception as e:
            raise CustomException(e, sys)

        with open(engine_path, "wb") as f:
            f.write(serialized_engine)
        logging.info(f"Engine saved to {engine_path}")
    
    def run(self) -> None:
        self._build_engine(
            onnx_path   = str(self.onnx_path),
            engine_path = str(self.engine_path),
        )

In [7]:
try:
    config = ConfigManager()
    tensorRT = TensorRT(config.get_tensorrt_config())
    tensorRT.run()
except Exception as e:
    raise CustomException(e, sys)

[2026-04-12 18:04:01,817: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-04-12 18:04:01,823: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-12 18:04:01,826: INFO: common: created directory at: artifacts]
[2026-04-12 18:04:01,836: INFO: common: created directory at: artifacts/tensorRT]
[2026-04-12 18:04:38,350: INFO: 957626049: Engine saved to artifacts/tensorRT/manet_resnet34/model.engine]


In [8]:
config = ConfigManager()
tensorRT = TensorRT(config.get_tensorrt_config())
tensorRT.quick_infer()

[2026-04-12 18:04:38,371: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-04-12 18:04:38,379: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-12 18:04:38,383: INFO: common: created directory at: artifacts]
[2026-04-12 18:04:38,387: INFO: common: created directory at: artifacts/tensorRT]
TensorRT:     6.56 ms
